# 🎙️ Voicebox: Qwen3-TTS Engine.
This uses [voicebox.sh](https://voicebox.sh) and the Qwen3 TTS engine. to generate Text to voice, then (optionally) save the output to your Google Drive.

This does REQUIRE you to have a voice profile, either from using the tool at the bottom of this notebook or [from an existing install of voicebox.sh](https://docs.voicebox.sh/overview/quick-start)

If you have a HUGE amount of text to convert to speech, This sytem may crash, It is HIGHLY recomended to use the save to Google Drive feature, as it will save each chunk it has processed. So you can just listen to the last section that was generated, and resume the process from there, and at the end use the merge feature.

At the end of each run it merges aLL sections in the output "Parts" Folder and merges them into the final output, not just the parts generated in that session.

In [ ]:
# @title 🎙️ Voicebox: Qwen3-TTS Direct Generator

# @markdown ### ⚙ **Model & Drive Settings** <br />
# @markdown Higher quality outputs take a litte longer to load, quality difference is marginal but noticable.
higher_quality_outputs = True # @param {type:"boolean"}
# @markdown Check this box to save to your Google Drive. <br />
# @markdown Uncheck to save temporarily in the Colab runtime it will automatically trigger a download of your data when it has finished.
save_to_drive = True # @param {type:"boolean"}
# @markdown Output will be saved under /Voicebox_Outputs/Project_name/
project_name = "Project_name" # @param ["Project_name"] {allow-input: true}

# @markdown ---
# @markdown ### 📝 **Text Input (Single Voice)**
# @markdown This is the text that will be turned to speech if Multi-Voice is disabled.<br />
text_to_speak = "Oh! Hello there! Wait, can you actually hear me?! Yay, I have a voice now!" # @param ["Oh! Hello there! Wait, can you actually hear me?! Yay, I have a voice now!"] {allow-input: true}

# @markdown ---
# @markdown ### 🎭 **Multi-Voice Support**
# @markdown Tick this box to upload a file instead of using the text box above. <br />
multi_voice_support = False # @param {type:"boolean"}
# @markdown **IN TESTING!!! Attempt to automatically Format my file for Multi Voice Mode. IN TESTING!!!!** <br />
# @markdown If checked, the AI will convert standard text into the necessary 'Character: "Speech"' play script format before generating voices.
auto_format_script = False # @param {type:"boolean"}
# @markdown **How it works:** You will be prompted to upload a `.txt` file. The file should contain multiple lines that must be formatted so that **each line STARTS with the character's name followed by a colon**. <br />
# @markdown When you run this cell it will prompt you for a voice profile for EACH of the characters that it finds in the file.<br />
# @markdown *Example:* <br />
# @markdown `Michael: "The only thing we're creating is a sugary disaster..."` <br />
# @markdown `Narrator: She reached out, her spiked collar catching the light...` <br />
# @markdown `Rajna: "I'm just making sure we don't accidentally....`

# @markdown ---
# @markdown ### 📁 **Voice Profile** <br />
# @markdown By default this will re-use the same voice profile for this whole session (Single Voice only). Untick this to be prompted for upload every time.
reuse_voice_profile = True # @param {type:"boolean"}

# @markdown ---
# @markdown ### Advanced Settings
# @markdown How many characters to process at once. Lower is more stable processing, Higher is better continuity in the tone of voice. <br />
# @markdown Note: *When using Multi Voice mode*, Each Line is sperate anyway, so this setting is PER LINE.
chunk_size = 2500 # @param {type:"slider", min:100, max:5000, step:50}

import sys
import os
import shutil
import zipfile
import json
import time
import re
import glob
from google.colab import files
from IPython.display import Audio, display, clear_output

# 1. Interactive Script & Upload Logic
script_lines = []
unique_characters = set()
voice_profiles = {}

if multi_voice_support:
    print("📝 Step 1/5: Multi-Voice Enabled. Please upload your script (.txt) file...")
    uploaded_script = files.upload()
    if not uploaded_script:
        raise FileNotFoundError("❌ No .txt script provided.")

    script_filename = list(uploaded_script.keys())[0]

    # Read the raw text safely
    try:
        raw_script_text = uploaded_script[script_filename].decode('utf-8')
    except UnicodeDecodeError:
        raw_script_text = uploaded_script[script_filename].decode('latin-1')

    # ---- AI FORMATTING LOGIC ----
    if auto_format_script:
        print("\n🤖 Auto-Formatting Script Enabled...")

        def run_ai_script_formatter(text_to_format):
            import gc
            import subprocess
            import torch

            print("📦 Ensuring text formatting dependencies are installed...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate"], check=True)

            from transformers import AutoModelForCausalLM, AutoTokenizer

            print("⏳ Loading Text Formatting Model (Qwen2.5-3B-Instruct) into GPU...")
            model_id = "Qwen/Qwen2.5-3B-Instruct"
            tokenizer = AutoTokenizer.from_pretrained(model_id)
            model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

            system_prompt = (
                "You are an expert scriptwriter. Your task is to convert narrative text into a strict play script format. "
                "Follow these rules perfectly:\n"
                "1. EVERY SINGLE LINE of your output MUST start with a speaker's name followed by a colon (e.g., `Character Name:` or `Narrator:`).\n"
                "2. Identify who is speaking and format dialogue as `Character Name: \"Spoken text\"`.\n"
                "3. ALL non-speech text (descriptive text, action tags, narrative exposition, e.g., 'he said', 'they walked away') MUST be explicitly assigned to the Narrator. Format this as `Narrator: Descriptive text`.\n"
                "4. If a sentence mixes dialogue and action, you MUST split them into separate lines. For example, '\"Hello,\" said John' becomes:\n"
                "   John: \"Hello,\"\n"
                "   Narrator: said John.\n"
                "5. Do not rewrite, change the story, or summarize. Keep the original words exactly.\n"
                "6. Output absolutely nothing else but the converted script. No conversational filler, titles, or introductions."
            )

            def chunk_text_semantically(input_text, max_chars=2500):
                sentences = re.split(r'(?<=[.!?\n])\s+', input_text)
                chunks, current_chunk = [], ""
                for sentence in sentences:
                    if len(current_chunk) + len(sentence) < max_chars:
                        current_chunk += sentence + " "
                    else:
                        if current_chunk.strip(): chunks.append(current_chunk.strip())
                        current_chunk = sentence + " "
                if current_chunk.strip(): chunks.append(current_chunk.strip())
                return chunks

            chunks = chunk_text_semantically(text_to_format, max_chars=2500)
            formatted_text = ""

            print(f"🧩 Document semantically split into {len(chunks)} chunks.")
            for i, chunk in enumerate(chunks):
                print(f" └─ Processing text chunk {i+1}/{len(chunks)}...")
                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"Convert the following text into the play script format:\n\n{chunk}"}
                ]
                formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                model_inputs = tokenizer([formatted_prompt], return_tensors="pt").to(model.device)

                generated_ids = model.generate(
                    **model_inputs,
                    max_new_tokens=2000,
                    temperature=0.3,
                    repetition_penalty=1.15,
                    do_sample=True,
                )
                generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
                output = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
                formatted_text += output + "\n\n"

            print("🧹 Cleaning up text model from VRAM to make room for TTS...")
            del model, tokenizer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            return formatted_text

        raw_script_text = run_ai_script_formatter(raw_script_text)
        print("✅ Formatting complete!\n")

        # Save a copy locally so the user can review it
        formatted_filename = f"Formatted_{script_filename}"
        with open(formatted_filename, 'w', encoding='utf-8') as f:
            f.write(raw_script_text)
        print(f"💾 Saved a copy of the AI formatted script locally as: {formatted_filename}")
    # -----------------------------

    # Parse the text (whether original or newly AI-formatted) line by line
    for line_num, line in enumerate(raw_script_text.splitlines(), 1):
        line = line.strip()
        if not line:
            continue
        if ':' in line:
            char_name, text = line.split(':', 1)
            char_name = char_name.strip()
            text = text.strip()
            if text:
                script_lines.append((char_name, text))
                unique_characters.add(char_name)
        else:
            print(f"⚠️ Warning: Skipping line {line_num} (No 'Charactername:' found) -> {line[:30]}...")

    print(f"\n🎭 Found {len(unique_characters)} characters: {', '.join(unique_characters)}")

    # Prompt for each character's voice profile
    for char in unique_characters:
        print(f"\n⬆️ Please upload the Voice Profile (.zip) for character: **{char}**")
        char_upload = files.upload()
        if char_upload:
            voice_profiles[char] = os.path.join('/content', list(char_upload.keys())[0])
        else:
            raise FileNotFoundError(f"❌ No .zip file provided for {char}.")
else:
    # Single voice logic
    script_lines.append(("Default", text_to_speak))
    unique_characters.add("Default")

    print("⬆️ Step 1/5: Please upload your Voicebox profile (.zip)")
    if reuse_voice_profile and 'global_profile_zip_path' in globals() and os.path.exists(global_profile_zip_path):
        print(f"♻️ Reusing previously uploaded voice profile: {global_profile_zip_path}")
        voice_profiles["Default"] = global_profile_zip_path
    else:
        uploaded = files.upload()
        if uploaded:
            uploaded_filename = list(uploaded.keys())[0]
            profile_zip_path = os.path.join('/content', uploaded_filename)
            global_profile_zip_path = profile_zip_path # Store globally for next run
            voice_profiles["Default"] = profile_zip_path
            print(f"✅ Uploaded successfully: {profile_zip_path}")
        else:
            raise FileNotFoundError("❌ No .zip file provided.")

# 2. Configure Output Path & Mount Drive (Interactive)
if save_to_drive:
    print("\n📂 Step 2/5: Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    base_output_dir = f'/content/drive/MyDrive/Voicebox_Outputs/{project_name}/'
else:
    print("\n📂 Step 2/5: Using local Colab runtime for saving...")
    base_output_dir = f'/content/Voicebox_Outputs/{project_name}/'

os.makedirs(base_output_dir, exist_ok=True)
print(f"✅ Output folder ready: {base_output_dir}")

# 3 & 4. System Dependencies, Repository Setup & Python Packages
if 'setup_completed' in globals() and setup_completed:
    print("\n✅ Steps 3 & 4: Dependencies and repository already set up in this session. Skipping...")
else:
    print("\n⏳ Step 3/5: Installing system packages and cloning Voicebox repository...")
    !apt-get install -y -qq sox libsox-fmt-all > /dev/null 2>&1
    !rm -rf /content/voicebox
    !git clone https://github.com/jamiepine/voicebox.git /content/voicebox

    print("\n📦 Step 4/5: Installing core Python dependencies (This takes a moment)...")
    !pip install -q -U qwen-tts accelerate soundfile transformers

    setup_completed = True

# ML Imports & Logging Configuration (Deferred until after pip install)
import torch
import numpy as np
import soundfile as sf
import transformers
import warnings

transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

from qwen_tts import Qwen3TTSModel

clear_output(wait=True)
print("✅ Voice profiles gathered.")
print(f"✅ Output folder ready: {base_output_dir}")

# 5. Load Qwen3-TTS Model
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-Base" if higher_quality_outputs else "Qwen/Qwen3-TTS-12Hz-0.6B-Base"
model_label = "1.7B" if higher_quality_outputs else "0.6B"

if 'model' in globals() and globals().get('current_model_id') == model_id:
    print(f"\n🧠 Step 5/5: Qwen3-TTS {model_label} Model is already loaded in VRAM. Skipping load.")
else:
    print(f"\n🧠 Step 5/5: Loading Qwen3-TTS {model_label} Model into GPU VRAM...")
    if 'model' in globals():
        del globals()['model']
        torch.cuda.empty_cache()

    target_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32

    model = Qwen3TTSModel.from_pretrained(
        model_id,
        torch_dtype=target_dtype,
        device_map="auto"
    )
    current_model_id = model_id
    print(f"✅ {model_label} Model loaded into GPU VRAM successfully!")

# 6. Text Chunking Helper
def chunk_text(text, max_chars=300):
    sentences = re.split(r'(?<=[.!?\n])\s+', text)
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_chars:
            current_chunk += sentence + " "
        else:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks

# 7. Unpack Voice Profiles
print("\n🎙️️ Unpacking Voice Profiles...")
char_references = {}
extracted_folders = []

try:
    for char, zip_path in voice_profiles.items():
        extract_path = f"/content/temp_voice_profile_{char.replace(' ', '_')}"
        extracted_folders.append(extract_path)
        os.makedirs(extract_path, exist_ok=True)

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)

        ref_audio = None
        ref_text = ""

        # Locate .wav audio
        for root, dirs, f_list in os.walk(extract_path):
            for file in f_list:
                if file.endswith(".wav") and not ref_audio:
                    ref_audio = os.path.join(root, file)

        # Locate transcript in samples.json
        if not ref_text and ref_audio:
            audio_basename = os.path.basename(ref_audio)
            for root, dirs, f_list in os.walk(extract_path):
                if "samples.json" in f_list:
                    try:
                        with open(os.path.join(root, "samples.json"), 'r', encoding='utf-8') as f:
                            data = json.load(f)
                            if isinstance(data, list):
                                for item in data:
                                    if isinstance(item, dict) and "text" in item:
                                        if "audio" in item and audio_basename in item["audio"]:
                                            ref_text = item["text"]
                                            break
                                        if not ref_text:
                                            ref_text = item["text"]
                    except Exception as e:
                        print(f"⚠️ Could not parse samples.json for {char}: {e}")

        if not ref_audio:
            raise FileNotFoundError(f"❌ No reference .wav audio found inside the .zip profile for {char}.")

        char_references[char] = {"audio": ref_audio, "text": ref_text}

    # 8. Generation Loop
    print("\n🎬 Generating Speech...")
    output_filename = os.path.join(base_output_dir, f"{project_name}_Finished.wav")
    chunks_dir = os.path.join(base_output_dir, f"{project_name}_Parts")
    os.makedirs(chunks_dir, exist_ok=True)

    # Determine starting part number
    existing_parts = glob.glob(os.path.join(chunks_dir, f"{project_name}_Part*.wav"))
    current_part_num = 0
    for p in existing_parts:
        filename = os.path.basename(p)
        match = re.search(rf'{re.escape(project_name)}_Part(\d+)\.wav$', filename)
        if match:
            num = int(match.group(1))
            if num > current_part_num:
                current_part_num = num

    print(f"📁 Chunks will be saved to: {chunks_dir}")
    print(f"🔢 Starting chunk numbering from Part{current_part_num + 1:04d}")

    for line_idx, (char, line_text) in enumerate(script_lines, 1):
        if not line_text.strip():
            continue

        print(f"\n🗣️ Character: {char} (Line {line_idx}/{len(script_lines)})")
        text_chunks = chunk_text(line_text, max_chars=chunk_size)

        ref_audio = char_references[char]["audio"]
        ref_text = char_references[char]["text"]

        for i, chunk in enumerate(text_chunks, 1):
            if not chunk.strip():
                continue
            print(f"   └─ Processing chunk {i}/{len(text_chunks)}...")

            if ref_text:
                wavs, sr = model.generate_voice_clone(
                    text=chunk,
                    language="English",
                    ref_audio=ref_audio,
                    ref_text=ref_text
                )
            else:
                wavs, sr = model.generate_voice_clone(
                    text=chunk,
                    language="English",
                    ref_audio=ref_audio,
                    x_vector_only_mode=True
                )

            chunk_audio = wavs[0]
            current_part_num += 1
            chunk_filename = os.path.join(chunks_dir, f"{project_name}_Part{current_part_num:04d}.wav")
            sf.write(chunk_filename, chunk_audio, sr)

    # 9. Combine ALL parts
    print(f"\n🔄 Scanning {chunks_dir} for all parts to combine...")
    all_parts = sorted(glob.glob(os.path.join(chunks_dir, f"{project_name}_Part*.wav")))

    if all_parts:
        print(f"🔗 Combining {len(all_parts)} parts...")
        combined_audio_arrays = []
        final_sample_rate = 24000 # Default fallback

        for part_file in all_parts:
            audio_data, sr = sf.read(part_file)
            combined_audio_arrays.append(audio_data)
            final_sample_rate = sr

        final_audio = np.concatenate(combined_audio_arrays)
        sf.write(output_filename, final_audio, final_sample_rate)

        print(f"\n🎉 Success! Combined audio saved to: {output_filename}\n")
        display(Audio(output_filename))

        # 10. Zip and Download logic if not saving to drive
        if not save_to_drive:
            print("\n📦 Zipping outputs for download...")
            zip_filename = f"/content/{project_name}_Output.zip"

            with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
                zipf.write(output_filename, arcname=os.path.basename(output_filename))
                for root, dirs, files_in_dir in os.walk(chunks_dir):
                    for file in files_in_dir:
                        file_path = os.path.join(root, file)
                        arcname = os.path.join(f"{project_name}_Parts", file)
                        zipf.write(file_path, arcname=arcname)

            print(f"⬇️ Downloading {os.path.basename(zip_filename)}...")
            files.download(zip_filename)
    else:
        print("❌ Generation failed or no audio parts were found to combine.")

finally:
    # Cleanup temporary folders
    for folder in extracted_folders:
        if os.path.exists(folder):
            shutil.rmtree(folder)

    # Cleanup zip files if reuse is false or multi-voice is enabled (to avoid clutter)
    if not reuse_voice_profile or multi_voice_support:
        for char, zip_path in voice_profiles.items():
            if os.path.exists(zip_path):
                os.remove(zip_path)
                print(f"🗑️ Cleaned up voice profile zip file: {zip_path}")
        if 'global_profile_zip_path' in globals():
            del globals()['global_profile_zip_path']

In [ ]:
# @title 📝 'Whisper' Subtitle Generator (.srt)

# @markdown ### ⚙ **Audio Input Settings**
# @markdown Select where the audio is coming from.
audio_source = "Auto-detect from previous cell" # @param ["Auto-detect from previous cell", "Upload a file", "Google Drive Path"]
# @markdown *(Only used if "Google Drive Path" is selected above)*
custom_audio_path = "/content/drive/MyDrive/Voicebox_Outputs/Project_name/Projct_Name_Finished.wav" # @param {type:"string"}

# @markdown ---
# @markdown ### 🧠 **Whisper Model Settings**
# @markdown Larger models are more accurate but take slightly longer and use more VRAM. That being said, I have had no issues with the largest.
whisper_model_size = "large-v3" # @param ["tiny", "base", "small", "medium", "large", "large-v3"]
language = "en" # @param {type:"string"}

# @markdown ---
# @markdown Note about how you will get the .srt file: <br />
# @markdown 1. If it came from Google drive, (Either via the previous cell, or manual path input) It will be saved alongside the input file. <br />
# @markdown 2. If it was a file upload it will immediatly offer it for download upon compleation.

import os
import subprocess
from google.colab import files
from IPython.display import clear_output

# 1. Install Dependencies if needed
try:
    import whisper
except ImportError:
    print("⏳ Installing Whisper and ffmpeg (This will just take a moment)...")
    subprocess.run(["pip", "install", "-q", "openai-whisper"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
    import whisper
    clear_output(wait=True)
    print("✅ Whisper installed successfully!")

# 2. Determine Audio Source
audio_file_to_process = None

if audio_source == "Auto-detect from previous cell":
    if 'output_filename' in globals() and os.path.exists(output_filename):
        audio_file_to_process = output_filename
        print(f"✅ Found audio from previous cell: {audio_file_to_process}")
    else:
        raise ValueError("❌ No audio found from a previous cell. Please run the generation cell first, or change 'audio_source' to Upload/Drive.")

elif audio_source == "Upload a file":
    print("⬆️ Please upload your audio file (.wav, .mp3, etc.)...")
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        audio_file_to_process = os.path.join('/content', filename)
        print(f"✅ Uploaded successfully: {audio_file_to_process}")
    else:
        raise FileNotFoundError("❌ No file uploaded.")

elif audio_source == "Google Drive Path":
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    if os.path.exists(custom_audio_path):
        audio_file_to_process = custom_audio_path
        print(f"✅ Found file at path: {audio_file_to_process}")
    else:
        raise FileNotFoundError(f"❌ File not found at path: {custom_audio_path}")

# 3. Helper function to format Whisper timestamps to SRT format
def format_timestamp(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds - int(seconds)) * 1000)
    return f"{hours:02}:{minutes:02}:{secs:02},{millis:03}"

def generate_srt_file(segments, output_path):
    with open(output_path, 'w', encoding='utf-8') as f:
        for i, segment in enumerate(segments, start=1):
            start = format_timestamp(segment['start'])
            end = format_timestamp(segment['end'])
            text = segment['text'].strip()
            f.write(f"{i}\n{start} --> {end}\n{text}\n\n")

# 4. Transcription and SRT Generation
if audio_file_to_process and os.path.exists(audio_file_to_process):
    print(f"\n🧠 Loading Whisper '{whisper_model_size}' model into VRAM...")

    # Check if model needs to be loaded/reloaded
    if 'current_whisper_model_size' not in globals() or current_whisper_model_size != whisper_model_size:
        import torch
        # Clean up VRAM if switching models
        if 'whisper_model' in globals():
            del globals()['whisper_model']
            torch.cuda.empty_cache()

        whisper_model = whisper.load_model(whisper_model_size)
        current_whisper_model_size = whisper_model_size
        print("✅ Whisper model loaded!")
    else:
        print("✅ Whisper model already loaded in this session.")

    print(f"\n🎧 Transcribing {os.path.basename(audio_file_to_process)} (this may take a minute)...")
    result = whisper_model.transcribe(audio_file_to_process, language=language)

    # 5. Output Logic
    base_name = os.path.splitext(os.path.basename(audio_file_to_process))[0]
    should_download = False

    if audio_source == "Upload a file":
        output_dir = '/content'
        should_download = True
    else:
        # If auto-detect was used or Drive path selected, save it in the same Drive directory
        audio_dir = os.path.dirname(audio_file_to_process)
        if '/content/drive' in audio_dir:
            output_dir = audio_dir
        else:
            from google.colab import drive
            if not os.path.exists('/content/drive'):
                drive.mount('/content/drive')

            # Use the project name from the previous cell if it exists, otherwise default folder
            p_name = globals().get('project_name', 'Standalone_Imports')
            output_dir = f'/content/drive/MyDrive/Voicebox_Outputs/{p_name}/'
            os.makedirs(output_dir, exist_ok=True)

    srt_output_path = os.path.join(output_dir, f"{base_name}.srt")

    print("\n📝 Formatting and saving .srt file...")
    generate_srt_file(result['segments'], srt_output_path)

    print(f"🎉 Success! SRT saved to: {srt_output_path}")

    if should_download:
        print("⬇️ Triggering download...")
        files.download(srt_output_path)

In [ ]:
# @title 🔗 Voicebox: Voice profile maker.

# Install pydub for actual audio conversion
!pip install -q pydub

import warnings
# Suppress the SyntaxWarning caused by pydub's outdated regex strings
warnings.filterwarnings("ignore", category=SyntaxWarning)

import os
import json
import uuid
import zipfile
from google.colab import files
from pydub import AudioSegment

# @markdown You can use this cell to make a voice profile for Cell 1. <br />
# @markdown You need four things: <br />
# @markdown 1. A name for the voice; this can be anything. <br />
# @markdown 2. A version number (optional, but helps you keep track!). <br />
# @markdown 3. A high-quality sample of the voice with no background noise. This should be around 20-30 seconds long. (It will prompt you to upload your voice sample when you run this cell).<br />
# @markdown 4. An EXACT WORD-FOR-WORD transcript of what is said in your example audio. Make this as complete as possible. Punctuate!? Add ... for pauses. If using non-words (like fantasy names), spell them as they SOUND. <br /> <br />
# @markdown Note: This can take almost any file type for the audio, it automatically coverts it to a .wav file, It can even take quite a few video file formats that it will extract the audio from!

# --- Interactive Colab Form Inputs ---
speaker_name = "My_Voice" # @param {type:"string"}
version = "v1" # @param {type:"string"}
transcript = "Type EXACTLY what is said in your sound file here." # @param {type:"string"}

print("Please upload your reference audio file:")
uploaded = files.upload()

if not uploaded:
    print("No file uploaded. Please run the cell again to upload a file.")
else:
    # 1. Prepare file paths
    original_audio_filename = list(uploaded.keys())[0]
    sample_uuid = str(uuid.uuid4())
    samples_dir = "samples"
    os.makedirs(samples_dir, exist_ok=True)
    new_audio_path = os.path.join(samples_dir, f"{sample_uuid}.wav")

    # 2. Actually convert the audio to WAV format
    print(f"Converting {original_audio_filename} to proper WAV format...")
    audio = AudioSegment.from_file(original_audio_filename)
    audio.export(new_audio_path, format="wav")

    # Clean up the original uploaded file
    os.remove(original_audio_filename)

    # 3. Generate the JSON metadata structures
    manifest_data = {
        "speaker": speaker_name,
        "version": version
    }

    samples_data = {
        f"samples/{sample_uuid}.wav": {
            "text": transcript
        }
    }

    # Write JSON files to disk
    with open("manifest.json", "w") as f:
        json.dump(manifest_data, f, indent=2)

    with open("manifest.json5", "w") as f:
        json.dump(manifest_data, f, indent=2)

    with open("samples.json", "w") as f:
        json.dump(samples_data, f, indent=2)

    # 4. Package everything into the ZIP archive
    formatted_name = speaker_name.lower().replace(" ", "-")
    zip_filename = f"profile-{formatted_name}-{version}.voicebox.zip"

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write("manifest.json")
        zipf.write("manifest.json5")
        zipf.write("samples.json")
        zipf.write(new_audio_path)

    print(f"\nSuccessfully packaged! Downloading {zip_filename}...")
    files.download(zip_filename)

    # 5. Cleanup temporary files from the Colab environment
    os.remove("manifest.json")
    os.remove("manifest.json5")
    os.remove("samples.json")
    os.remove(new_audio_path)
    os.rmdir(samples_dir)